In [41]:
import numpy as np
import pandas
import sklearn.compose
import sklearn.preprocessing
import sklearn.model_selection

from lazyfca import LazyFCA

from utils import estimate_quality

In [42]:
data = pandas.read_csv("churn.csv")
data = data.drop(columns = ['customerID'])
data = data[data["TotalCharges"] != ' ']
data["TotalCharges"] = data["TotalCharges"].astype(float)

cols_to_replace = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
data[cols_to_replace] = data[cols_to_replace].replace(['No phone service', 'No internet service'], 'No')

X = data.drop(columns = ["Churn"])
# y = data["Churn"].to_numpy()
y = (data["Churn"] == "Yes").to_numpy()
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size = 0.1, stratify = y, random_state = 42
)

numeric = [ "tenure", "MonthlyCharges", "TotalCharges" ]
categorical = list(set(X_train.columns) - set(numeric))
ct = sklearn.compose.ColumnTransformer(
    transformers = [
        ("numeric", 'passthrough', numeric),
        ("categorical", sklearn.preprocessing.OneHotEncoder(dtype = 'bool'), categorical)
    ]
)
X_train = pandas.DataFrame(ct.fit_transform(X_train), columns = ct.get_feature_names_out())
X_test = pandas.DataFrame(ct.transform(X_test), columns = ct.get_feature_names_out())

categorical = [ feature for feature in ct.get_feature_names_out() if feature.startswith("categorical__") ]
X_train[categorical] = X_train[categorical].astype(bool)
X_test[categorical] = X_test[categorical].astype(bool)

y_train = pandas.Series(y_train)
y_test = pandas.Series(y_test)

In [43]:
classifier = LazyFCA(
    min_pos_for_pos_clas = 5,
    pos_coef_for_pos_clas = 2.75,
    min_neg_for_neg_clas = 10,
    neg_coef_for_neg_clas = 0.25,
    pos_clas_coef = 1.0
)
classifier.fit(X_train, y_train)

In [44]:
y_pred = classifier.predict(X_test)

# estimate_quality(y_pred, y_test)













































































































































100%|██████████| 704/704 [01:28<00:00,  7.94it/s]


In [5]:
classifier.explain(X_test.iloc[502]).display()

,Type,Supporters,Opposers,Supporters covered,Opposers covered
0,POSITIVE,1682,4646,5,6
1,POSITIVE,1682,4646,7,14
2,POSITIVE,1682,4646,341,874
3,POSITIVE,1682,4646,6,12
4,POSITIVE,1682,4646,5,13
...,...,...,...,...,...
3547,NEGATIVE,4646,1682,244,26
3548,NEGATIVE,4646,1682,25,0
3549,NEGATIVE,4646,1682,32,0
3550,NEGATIVE,4646,1682,74,1


# 1. Coverage & purity + 3. Imbalance-aware utility

In [6]:
base_metrics = estimate_quality(y_pred, y_test)['total']
base_metrics

{'Accuracy': 0.7556818181818182,
 'Precision': 0.5265017667844523,
 'Recall': 0.7967914438502673,
 'AUC-ROC': 0.8240724459293125,
 'F1-score': 0.6340425531914894,
 'True Positive': np.int64(149),
 'True Negative': np.int64(383),
 'False Positive': np.int64(134),
 'False Negative': np.int64(38),
 'True Negative Rate (Specificity)': np.float64(0.7408123791102514),
 'Negative Predictive Value': np.float64(0.9097387173396675),
 'False Positive Rate': np.float64(0.25918762088974856),
 'False Discovery Rate': np.float64(0.4734982332155477),
 'Balanced precision proxy': np.float64(0.5376038229605188),
 "Youden's J": np.float64(0.5376038229605188),
 'Matthews correlation': 0.4842773502810908}

- not sure about Balanced precision proxy
- Youden's J = 0.54 => our classifier not random (0) but not so perfect (1)
- Matthews correlation = 0.48 => our classifier not random (0) but not so perfect (1)

Look the metrics for positives hypothesis. Compute from total metrics

In [ ]:
# Positive class h+
tp = base_metrics['True Positive']
fp = base_metrics['False Positive']
P = np.sum(y_test)
N = np.sum(y_test == 0)

# supp(h+) = tp / P  (= Recall)
support = base_metrics['Recall']

# cont(h+) = fp / N  (= False Positive Rate)
contamination = fp / N

# prec(h+) = tp / (tp + fp)  (= Precision)
precision = base_metrics['Precision']

# lift
lift = precision / (P / (P + N))

# WRAcc(h+)
wracc = ((tp + fp) / (P + N)) * (precision - (P / (P + N)))

print(f'Support (positive coverage): {support}')
print(f'Contamination (error rate): {contamination}')
print(f'Precision (confidence): {precision}')
print(f'Lift: {lift}')
print(f'WRAcc: {wracc}')

Support (positive coverage): 0.7967914438502673
Contamination (error rate): 0.25918762088974856
Precision (confidence): 0.5265017667844523
Lift: 1.9821242984826437
WRAcc: 0.10486949573863635


- Support seems fine for the positive class because is less in objects
- Error rate is 25%
- precision is very small, i suppose, because it is imbalanced
- lift 1.98 > 1 shows us that our model in 2 times more effisient than random classifier
- WRAcc = 0.1 > 0 shows us that positive classifier have a nice connection to positive objeccts overall

I have decided to code this metric for every class of hypothesis (NEED VALIDATION)

In [8]:
# metrics for positive hypothesis
estimate_quality(y_pred, y_test)['positive']

{'Support': np.float64(0.7967914438502673),
 'Contamination': np.float64(0.0),
 'Confidence': np.float64(1.0),
 'Lift': np.float64(3.764705882352941),
 'WRAcc': np.float64(0.1554287997159091),
 'Balanced precision proxy': np.float64(0.7967914438502673),
 'Matthews correlation': np.float64(0.7682922394332828)}

- Support seems fine for the positive class because is less in objects
- Error rate is 0% because I choose only positive objects
- precision is max for the same reason
- lift 3.76 > 1 shows us that our model in 3.8 times more effisient than random classifier (is it valid conclusion?)
- WRAcc = 0.16 > 0 and close to 0.25 shows us that positive classifier have a very nice connection to positive objeccts overall
- Balanced precision proxy = 0.8 => close to perfect classification in imbalanced dataset
- not sure about Matthews correlation

In [9]:
# metrics for negative hypothesis
estimate_quality(y_pred, y_test)['negative']

{'Support': np.float64(0.7408123791102514),
 'Contamination': np.float64(0.0),
 'Confidence': np.float64(1.0),
 'Lift': np.float64(1.3617021276595744),
 'WRAcc': np.float64(0.1445090553977273),
 'Balanced precision proxy': np.float64(0.7408123791102514),
 'Matthews correlation': 0}

- Support seems fine for the negative class
- Error rate is 0% because I choose only negative objects
- precision is max for the same reason
- lift 1.36 > 1 shows us that our model in 1.4 times more effisient than random classifier (is it valid conclusion?)
- WRAcc = 0.15 > 0 and close to 0.25 shows us that negative classifier have a very nice connection to negative objeccts overall
- Balanced precision proxy = 0.74 => close to perfect classification in imbalanced dataset
- not sure about Matthews correlation

## 5. Stability/robustness

I slightly change one object and see the results

In [23]:
idx = 502
example = X_test.iloc[idx]
example.to_frame().T

,numeric__tenure,numeric__MonthlyCharges,numeric__TotalCharges,categorical__OnlineBackup_No,categorical__OnlineBackup_Yes,categorical__TechSupport_No,categorical__TechSupport_Yes,categorical__gender_Female,categorical__gender_Male,categorical__InternetService_DSL,...,categorical__StreamingMovies_No,categorical__StreamingMovies_Yes,categorical__OnlineSecurity_No,categorical__OnlineSecurity_Yes,categorical__PaymentMethod_Bank transfer (automatic),categorical__PaymentMethod_Credit card (automatic),categorical__PaymentMethod_Electronic check,categorical__PaymentMethod_Mailed check,categorical__DeviceProtection_No,categorical__DeviceProtection_Yes
502,72.0,19.55,1463.45,True,False,True,False,True,False,False,...,True,False,True,False,True,False,False,False,True,False


In [24]:
y_pred = classifier.predict(example.to_frame().T)
y_pred


100%|██████████| 1/1 [00:00<00:00, 692.59it/s]


array([[0.99802928, 0.00197072]])

In [25]:
y_test.iloc[idx]

np.False_

In [26]:
classifier.explain(example).display()

,Type,Supporters,Opposers,Supporters covered,Opposers covered
0,POSITIVE,1682,4646,5,6
1,POSITIVE,1682,4646,7,14
2,POSITIVE,1682,4646,341,874
3,POSITIVE,1682,4646,6,12
4,POSITIVE,1682,4646,5,13
...,...,...,...,...,...
3547,NEGATIVE,4646,1682,244,26
3548,NEGATIVE,4646,1682,25,0
3549,NEGATIVE,4646,1682,32,0
3550,NEGATIVE,4646,1682,74,1


Let's mess with some features

In [34]:
example_mess = example.copy().to_frame().T
example_mess['numeric__tenure'] = 10
example_mess['numeric__MonthlyCharges'] = 100
example_mess['numeric__TotalCharges'] = 500
example_mess

,numeric__tenure,numeric__MonthlyCharges,numeric__TotalCharges,categorical__OnlineBackup_No,categorical__OnlineBackup_Yes,categorical__TechSupport_No,categorical__TechSupport_Yes,categorical__gender_Female,categorical__gender_Male,categorical__InternetService_DSL,...,categorical__StreamingMovies_No,categorical__StreamingMovies_Yes,categorical__OnlineSecurity_No,categorical__OnlineSecurity_Yes,categorical__PaymentMethod_Bank transfer (automatic),categorical__PaymentMethod_Credit card (automatic),categorical__PaymentMethod_Electronic check,categorical__PaymentMethod_Mailed check,categorical__DeviceProtection_No,categorical__DeviceProtection_Yes
502,10,100,500,True,False,True,False,True,False,False,...,True,False,True,False,True,False,False,False,True,False


In [36]:
y_pred = classifier.predict(example_mess)
y_pred


100%|██████████| 1/1 [00:00<00:00, 634.06it/s]


array([[0.64030612, 0.35969388]])

As we can see classifier less sure about negative class but still predict as negative despite I changed all 3 main features